<a href="https://colab.research.google.com/github/BernardoBremer/Inteligencia-Computacional-Cetys-/blob/main/9_vectorstore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector stores and semantic search



## Part I: Basic vector store implementation

In [21]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [23]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeddings = self.embedding_model.encode(texts, show_progress_bar=False)
        self.documents.extend(documents)
        if self.embeddings is None:
            self.embeddings = np.array(new_embeddings)
        else:
            self.embeddings = np.vstack([self.embeddings, np.array(new_embeddings)])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        query_embedding = self.embedding_model.encode([query])[0]
        norms = np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(query_embedding)
        similarities = np.dot(self.embeddings, query_embedding) / norms
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [SearchResult(score=float(similarities[i]), document=self.documents[i]) for i in top_indices]

In [24]:
df = pd.read_csv("animal-fun-facts-dataset.csv")
df = df.fillna("")

documents = []
for _, row in df.iterrows():
    documents.append(Document(
        text=str(row["text"]),
        metadata={
            "animal_name": str(row["animal_name"]),
            "source": str(row["source"]),
            "media_link": str(row["media_link"]),
            "wikipedia_link": str(row["wikipedia_link"])
        }
    ))

print(f"Documentos cargados: {len(documents)}")

store = VectorStore(model)
store.add_documents(documents)

Documentos cargados: 7734


In [25]:
def mostrar_resultados(query, results):
    print(f"Consulta: '{query}'")
    print()
    for i, r in enumerate(results):
        print(f"  Resultado {i+1}:")
        print(f"    Score: {r.score:.4f}")
        print(f"    Texto: {r.document.text[:200]}")
        print(f"    Metadatos: {r.document.metadata}")
        print()

consultas = [
    "animals that can fly very fast",
    "venomous snakes and their dangerous bite",
    "deep sea ocean creatures bioluminescence",
    "largest animals in the world by weight",
    "endangered species and conservation efforts"
]

for q in consultas:
    results = store.search(q, top_k=3)
    mostrar_resultados(q, results)
    print()

Consulta: 'animals that can fly very fast'

  Resultado 1:
    Score: 0.7108
    Texto: Fastest animal on Earth
    Metadatos: {'animal_name': 'peregrine falcon', 'source': 'https://a-z-animals.com/animals/peregrine-falcon/', 'media_link': '', 'wikipedia_link': '/wiki/Peregrine_falcon'}

  Resultado 2:
    Score: 0.6832
    Texto: The fastest creatures on the planet!
    Metadatos: {'animal_name': 'falcon', 'source': 'https://a-z-animals.com/animals/falcon/', 'media_link': '', 'wikipedia_link': '/wiki/Falcon'}

  Resultado 3:
    Score: 0.6659
    Texto: They are the fastest bird in the world.
The peregrine falcon’s streamlined body and pointed wings allow it to reach high speeds in flight and when diving to hunt prey.
    Metadatos: {'animal_name': 'peregrine falcon', 'source': 'https://factanimal.com/peregrine-falcon/', 'media_link': '', 'wikipedia_link': '/wiki/Peregrine_falcon'}


Consulta: 'venomous snakes and their dangerous bite'

  Resultado 1:
    Score: 0.8487
    Texto: The 

## Part II: Filtering by metadata

In [26]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeddings = self.embedding_model.encode(texts, show_progress_bar=False)
        self.documents.extend(documents)
        if self.embeddings is None:
            self.embeddings = np.array(new_embeddings)
        else:
            self.embeddings = np.vstack([self.embeddings, np.array(new_embeddings)])

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        query_embedding = self.embedding_model.encode([query])[0]
        if metadata_filter:
            indices = [i for i, doc in enumerate(self.documents)
                       if all(doc.metadata.get(k) == v for k, v in metadata_filter.items())]
        else:
            indices = list(range(len(self.documents)))
        if not indices:
            return []
        filtered_embeddings = self.embeddings[indices]
        norms = np.linalg.norm(filtered_embeddings, axis=1) * np.linalg.norm(query_embedding)
        similarities = np.dot(filtered_embeddings, query_embedding) / norms
        top_k_adj = min(top_k, len(indices))
        top_positions = np.argsort(similarities)[::-1][:top_k_adj]
        results = []
        for pos in top_positions:
            results.append(SearchResult(
                score=float(similarities[pos]),
                document=self.documents[indices[pos]]
            ))
        return results

Dataset seleccionado: 20 Newsgroups Este dataset contiene publicaciones de grupos de noticias de Usenet organizadas por tematica, Cada documento tiene como metadatos el nombre del grupo (newsgroup) y el tema general (topic).

In [27]:
from sklearn.datasets import fetch_20newsgroups

categorias = [
    "rec.sport.baseball",
    "sci.space",
    "comp.graphics",
    "talk.politics.mideast",
    "misc.forsale"
]

newsgroups = fetch_20newsgroups(
    subset="train",
    categories=categorias,
    remove=("headers", "footers", "quotes")
)

news_documents = []
for text, target in zip(newsgroups.data, newsgroups.target):
    clean_text = text.strip()
    if len(clean_text) < 20:
        continue
    newsgroup_name = newsgroups.target_names[target]
    topic = newsgroup_name.split(".")[0]
    news_documents.append(Document(
        text=clean_text,
        metadata={
            "newsgroup": newsgroup_name,
            "topic": topic
        }
    ))

print(f"Documentos cargados: {len(news_documents)}")

filtered_store = FilteredVectorStore(model)
filtered_store.add_documents(news_documents)

Documentos cargados: 2830


In [28]:
def mostrar_resultados_filtrados(query, filtro, results):
    print(f"Consulta: '{query}'")
    print(f"Filtro: {filtro}")
    print()
    for i, r in enumerate(results):
        print(f"  Resultado {i+1}:")
        print(f"    Score: {r.score:.4f}")
        print(f"    Texto: {r.document.text[:200]}")
        print(f"    Metadatos: {r.document.metadata}")
        print()

consultas_filtradas = [
    ("pitcher throws fastball strikeout", {"newsgroup": "rec.sport.baseball"}),
    ("NASA space shuttle launch orbit", {"newsgroup": "sci.space"}),
    ("3D image rendering algorithm", {"newsgroup": "comp.graphics"}),
    ("conflict territory peace negotiations", {"newsgroup": "talk.politics.mideast"}),
    ("selling used car good condition price", {"topic": "misc"})
]

for q, filtro in consultas_filtradas:
    results = filtered_store.search(q, top_k=3, metadata_filter=filtro)
    mostrar_resultados_filtrados(q, filtro, results)
    print()

Consulta: 'pitcher throws fastball strikeout'
Filtro: {'newsgroup': 'rec.sport.baseball'}

  Resultado 1:
    Score: 0.5395
    Texto: Kevin Mitchell's sacrifice fly in the eighth off Brett Saberhagen plated 
pitch runner Cesar Hernandez to give the Reds a 2-3 come-from-behind victory over 
New York. Hernandez ran for pinch-hitter Ce
    Metadatos: {'newsgroup': 'rec.sport.baseball', 'topic': 'rec'}

  Resultado 2:
    Score: 0.5264
    Texto: are you serious? pitchers are pinch-hit for in the nl.  they are not in the
nl.  if a pitcher is cranking in the al, he will stay in the game.  if he
is cranking in the nl, he may not - ESPECIALLY if 
    Metadatos: {'newsgroup': 'rec.sport.baseball', 'topic': 'rec'}

  Resultado 3:
    Score: 0.5102
    Texto: Despite walks and loses, Ryan deserves to be in the Hall of Fame (IMHO)
based only on his ho-hitters.  The strike-out records are an extra.

What do people think about Andre "400 HR" Dawson for the HO
    Metadatos: {'newsgroup': 'rec.spor

#reflexiones

No pude asistir a las ultimas clases por lo del hackaton, asi que tuve que investigar por mi cuenta. Entre una respuesta de IA y las diapositivas del curso pude entender como funcionan los embeddings.

Lo que mas me llamo la atencion es que la busqueda semantica encuentra documentos relevantes sin necesidad de que coincidan las palabras exactas, el modelo captura el significado detras del texto y eso hace que funcione mucho mejor que una busqueda por keywords (como se vio en una clase).

El FilteredVectorStore fue mas rapido, la logica es filtrar primero por metadatos y despues hacer la busqueda sobre ese subconjunto.es mas  sencillo que funciona bien para datasets de este tamaño.